In [ ]:
%%bash
# Chẩn đoán: GPU, Python, thư viện có sẵn của image, cấu trúc các dataset đã gắn
nvidia-smi --query-gpu=index,name,memory.total --format=csv
which python; python --version
echo "== python -I (chế độ cô lập) có thấy torch không?"
python -I -c "import torch; print('ok', torch.__version__)" 2>&1 | tail -1
echo "== thư viện (chế độ thường)"
python - <<'PY'
import importlib.util as u, torch
print("torch", torch.__version__, "| gpus", torch.cuda.device_count())
for m in ["transformers", "peft", "accelerate", "jiwer", "jsonschema", "soundfile", "librosa", "datasets"]:
    print(f"{m:14s}", "OK" if u.find_spec(m) else "MISSING")
import transformers
print("transformers", transformers.__version__)
PY
echo "== các dataset đã gắn"
for d in /kaggle/input/*; do echo "-- $d"; ls "$d" | head -6; done
df -h /kaggle/working | tail -1

In [ ]:
# KIỂM TRA DATASET ĐÃ UPLOAD (chạy được ngay, không cần cell setup). Chỉ đọc, không ghi gì.
# Bật DO_SHA = True để băm sha256 từng shard (chậm vài phút, bắt lỗi hỏng nội dung); LOAD_TEST = True để thử nạp thật Qwen3-4B.
import hashlib, json, os, struct, time
from collections import defaultdict
from pathlib import Path

DO_SHA = os.environ.get("CHECK_SHA", "0") == "1"
LOAD_TEST = os.environ.get("CHECK_LOAD", "1") == "1"
IN = Path(os.environ.get("CHECK_IN", "/kaggle/input"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")

# ---- giá trị chuẩn lấy từ máy local (docs/MODELS.md, kaggle_upload/lifemate-models/manifest.json)
QWEN = {"config.json": 726, "generation_config.json": 239, "merges.txt": 1671853, "tokenizer.json": 11422654,
        "tokenizer_config.json": 9732, "vocab.json": 2776833, "model.safetensors.index.json": 32819,
        "model-00001-of-00003.safetensors": 3957900840, "model-00002-of-00003.safetensors": 3987450520,
        "model-00003-of-00003.safetensors": 99630640}
PHO = {"config.json": 1272, "generation_config.json": 4573, "merges.txt": 493864, "tokenizer.json": 3654002,
       "tokenizer_config.json": 21520, "vocab.json": 835550, "added_tokens.json": 2190, "normalizer.json": 54408,
       "preprocessor_config.json": 371, "special_tokens_map.json": 2333, "model.safetensors.index.json": 112901,
       "model-00001-of-00002.safetensors": 4992706480, "model-00002-of-00002.safetensors": 1180663192}
SHA = {"model-00001-of-00003.safetensors": "328a91d3122359d5547f9d79521205bc0a46e1f79a792dfe650e99fc2d651223",
       "model-00002-of-00003.safetensors": "6cd087b316306a68c562436b5492edbcf6e16c6dba3a1308279caa5a58e21ca5",
       "model-00003-of-00003.safetensors": "e4bf436957184f4eeb86a80e9db394503f1f56446b2e6b7edeac5b81470f4ca1",
       "model-00001-of-00002.safetensors": "4855e8b43394e0378a0fbc85945f7539f9ce0196ef568548d44b3555bd24d59c",
       "model-00002-of-00002.safetensors": "f06ccdf86a8b2260267a1bb5b1db17855979e220d85688377ab7eefb84017de2"}
PROCESSED = {"dev_ids.txt": 1860, "sft_train.jsonl": 10224065, "sft_dev.jsonl": 1112339}
CODE_FILES = ["src/lifemate/agent.py", "src/lifemate/prompt.py", "scripts/kaggle_setup.sh", "scripts/kaggle_pipeline.sh",
              "scripts/infer_multi_gpu.sh", "scripts/merge_shards.py", "scripts/train_sft.py", "scripts/train_asr.py",
              "scripts/run_asr.py", "scripts/run_infer.py", "requirements-kaggle.txt"]

problems = []


def say(ok, msg, hard=True):
    print(("[ OK ] " if ok else ("[LỖI] " if hard else "[cảnh báo] ")) + msg)
    if not ok and hard:
        problems.append(msg)


# Quét /kaggle/input MỘT lần (bỏ qua thư mục competitions và không mở từng file wav), mọi bước sau tra cứu trong chỉ mục này.
FILES = defaultdict(list)      # tên file -> [thư mục chứa]
DIRS = defaultdict(list)       # tên thư mục -> [đường dẫn]
WAVS = defaultdict(int)        # (thư mục gốc dữ liệu, split) -> số file wav
_t0 = time.time()
_nfiles = 0
for dp, dn, fn in os.walk(IN):
    d = Path(dp)
    # bỏ qua cuộc thi khác và các thư mục trạng thái app của đề (luật: không mở/liệt kê); chỉ cần train.jsonl, audio, catalog...
    dn[:] = [x for x in dn if not (d == IN and x == "competitions") and not x.endswith("_environment")]
    for x in dn:
        DIRS[x].append(d / x)
    if d.name == "audio" or "audio" in d.parts:
        i = d.parts.index("audio")
        WAVS[(Path(*d.parts[: i - 1]), d.parts[i - 1])] += sum(1 for f in fn if f.endswith(".wav"))
        _nfiles += len(fn)
        continue
    for f in fn:
        FILES[f].append(d)
    _nfiles += len(fn)
print(f"Đã quét {_nfiles} file trong {time.time() - _t0:.1f} giây")


def find_by_name(name):
    return [str(d / name) for d in FILES.get(name, [])][:3]


def nlines(p):
    with open(p, "rb") as f:
        return sum(1 for _ in f)


def safetensors_problem(p):
    """Đối chiếu độ dài thực với độ dài khai báo trong header safetensors: bắt file bị cắt/upload dở."""
    size = p.stat().st_size
    with open(p, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        if n > 200_000_000:
            return "header safetensors không hợp lệ (file hỏng)"
        header = json.loads(f.read(n))
    end = max((v["data_offsets"][1] for k, v in header.items() if k != "__metadata__"), default=0)
    return None if 8 + n + end == size else f"dài {size} byte nhưng header yêu cầu {8 + n + end} (file bị cắt hoặc upload dở)"


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()


def model_dir(model_type):
    for d in FILES.get("config.json", []):
        try:
            if json.loads((d / "config.json").read_text(encoding="utf-8")).get("model_type") == model_type:
                return d
        except Exception:
            pass
    return None


def check_model(label, model_type, expected):
    print(f"\n=== {label}")
    d = model_dir(model_type)
    if d is None:
        say(False, f"{label}: không thấy thư mục nào có config.json với model_type={model_type}. "
                   f"Dataset model chưa được gắn (Add Input) hoặc chưa upload config.json")
        return None
    print("thư mục:", d)
    for name, size in expected.items():
        p = d / name
        if not p.exists():
            say(False, f"THIẾU {name}" + (f" (nhưng có ở: {find_by_name(name)})" if find_by_name(name) else " (không thấy ở đâu trong /kaggle/input)"))
            continue
        got = p.stat().st_size
        if got != size:
            say(False, f"{name}: dài {got} byte, chuẩn {size} (thiếu {size - got})")
            continue
        extra = ""
        if name.endswith(".safetensors"):
            pr = safetensors_problem(p)
            if pr:
                say(False, f"{name}: {pr}")
                continue
            if DO_SHA:
                ok = sha256(p) == SHA[name]
                say(ok, f"{name}: {got/1e9:.2f} GB, sha256 {'khớp' if ok else 'KHÔNG KHỚP (nội dung hỏng)'}")
                continue
            extra = f" ({got/1e9:.2f} GB, header safetensors khớp)"
        say(True, f"{name}{extra}")
    idx = d / "model.safetensors.index.json"
    if idx.exists():
        shards = set(json.loads(idx.read_text(encoding="utf-8"))["weight_map"].values())
        miss = [s for s in shards if not (d / s).exists()]
        say(not miss, f"index.json tham chiếu {len(shards)} shard" + (f", THIẾU {miss}" if miss else ", đều có mặt"))
    return d


print("Các mục dưới /kaggle/input:", sorted(str(p.relative_to(IN)) for p in IN.glob("*/*"))[:12] or "(trống)")

# ---------------------------------------------------------------- code
print("\n=== lifemate-code")
codes = [d for d in FILES.get("agent.py", []) if d.name == "lifemate" and d.parent.name == "src"]
if not codes:
    say(False, "không thấy src/lifemate/agent.py: dataset code chưa gắn hoặc zip lồng thêm thư mục quá sâu")
else:
    root = codes[0].parent.parent
    print("thư mục:", root)
    for rel in CODE_FILES:
        say((root / rel).exists(), rel, hard=True)

# ---------------------------------------------------------------- raw
print("\n=== dữ liệu cuộc thi (raw)")
trains = [d for d in FILES.get("train.jsonl", []) if d.name == "train"]
if not trains:
    say(False, "không thấy train/train.jsonl: dataset raw chưa gắn hoặc chưa giải nén")
else:
    raw = trains[0].parent
    print("thư mục:", raw)
    for sp, wavs, lines in [("train", 5036, 1545), ("public_test", 1933, 634)]:
        j, cat = raw / sp / f"{sp}.jsonl", raw / sp / f"{sp}_tools_catalog.json"
        say(j.exists() and nlines(j) == lines, f"{sp}.jsonl: {nlines(j) if j.exists() else 'thiếu'} dòng (chuẩn {lines})")
        say(cat.exists(), f"{sp}_tools_catalog.json")
        n = WAVS.get((raw, sp), 0)
        say(n == wavs, f"{sp}/audio: {n} file wav (chuẩn {wavs})")
    say((raw / "train" / "zaic_tools").exists(), "train/zaic_tools")

# ---------------------------------------------------------------- processed
print("\n=== lifemate-processed")
_dev = FILES.get("dev_ids.txt", [])
dev = (_dev[0] / "dev_ids.txt") if _dev else None
if dev is None:
    say(False, "không thấy dev_ids.txt: dataset processed chưa gắn")
else:
    for name, size in PROCESSED.items():
        p = dev.parent / name
        say(p.exists() and p.stat().st_size == size, f"{name}: {p.stat().st_size if p.exists() else 'thiếu'} byte (chuẩn {size})")
    say(nlines(dev) == 155, f"dev_ids.txt có {nlines(dev)} id (chuẩn 155)")

# ---------------------------------------------------------------- models
qdir = check_model("Qwen3-4B", "qwen3", QWEN)
pdir = check_model("PhoWhisper-large", "whisper", PHO)

# ---------------------------------------------------------------- libs + wheelhouse
print("\n=== thư viện và wheelhouse")
import importlib.util as iu
for m in ["torch", "transformers", "peft", "accelerate", "jiwer", "jsonschema", "soundfile", "librosa", "datasets"]:
    say(iu.find_spec(m) is not None, f"import {m}" + ("" if iu.find_spec(m) else " (thiếu; cần dataset lifemate-wheels hoặc image có sẵn)"),
        hard=m in ("torch", "transformers", "peft"))
try:
    import transformers
    v = tuple(int(x) for x in transformers.__version__.split(".")[:2])
    say(v >= (4, 56), f"transformers {transformers.__version__} (các script dùng dtype= nên cần >= 4.56)")
except Exception as e:
    say(False, f"transformers: {e}")
wh = DIRS.get("wheelhouse", [])
say(bool(wh), "dataset lifemate-wheels (thư mục wheelhouse)" + (f": {len(list(wh[0].iterdir()))} file" if wh else " chưa gắn"), hard=False)

# ---------------------------------------------------------------- nạp thật
if LOAD_TEST and qdir is not None:
    print("\n=== thử nạp Qwen3-4B (tái hiện đúng lỗi nếu có)")
    try:
        import torch
        from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
        cfg = AutoConfig.from_pretrained(qdir)
        tok = AutoTokenizer.from_pretrained(qdir)
        say(True, f"config + tokenizer: {tok('xin chào các bạn')['input_ids']}")
        _t = time.time()
        model = AutoModelForCausalLM.from_pretrained(qdir, dtype=torch.bfloat16, **({"device_map": "cuda"} if torch.cuda.is_available() else {}))
        say(True, f"nạp model lên {'GPU' if torch.cuda.is_available() else 'CPU'} thành công trong {time.time() - _t:.0f} giây, "
                  f"{sum(p.numel() for p in model.parameters()) / 1e9:.3f}B tham số")
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    except Exception as e:
        say(False, f"nạp Qwen3-4B lỗi: {type(e).__name__}: {str(e)[:300]}")

# ---------------------------------------------------------------- kết luận
print("\n" + "=" * 70)
if problems:
    print(f"KẾT LUẬN: {len(problems)} vấn đề. Sửa lần lượt (thường là upload thiếu/dở file hoặc quên Add Input):")
    for i, p in enumerate(problems, 1):
        print(f" {i}. {p}")
else:
    print("KẾT LUẬN: tất cả dataset đủ và đúng. Có thể chạy cell setup.")

In [ ]:
%%bash
# Setup: không phụ thuộc cách đặt tên/lồng thư mục của dataset model (tìm theo model_type trong config.json)
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
QW=$(grep -rlE --include=config.json '"model_type"[[:space:]]*:[[:space:]]*"qwen3"' /kaggle/input 2>/dev/null | head -1 | xargs -r dirname)
PH=$(grep -rlE --include=config.json '"model_type"[[:space:]]*:[[:space:]]*"whisper"' /kaggle/input 2>/dev/null | head -1 | xargs -r dirname)
echo "Qwen3 dir  : ${QW:-KHÔNG THẤY}"
echo "Whisper dir: ${PH:-KHÔNG THẤY}"
[ -n "$QW" ] && [ -n "$PH" ] || { echo "ERROR: thiếu thư mục model, kiểm tra hai dataset model đã gắn chưa"; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
# dùng thư mục model đã dò được; bỏ chế độ cô lập -I (image Python 3.13 có thể giữ thư viện ngoài sys.path cô lập)
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e "s#^PHO=.*#PHO=$PH#" -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
# cùng lý do: bỏ -I trong bản làm việc của các script (không đụng dataset code gốc)
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "còn lại 'python -I' trong scripts/*.sh: $(grep -c 'python -I ' /kaggle/working/lifemate/scripts/*.sh | awk -F: '{s+=$2} END {print s}')"
cat /kaggle/working/lifemate/kaggle_env.sh

In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh || { echo "ERROR: chưa có kaggle_env.sh, chạy cell setup trước"; exit 1; }
echo "QWEN=$QWEN"; echo "PHO=$PHO"
ls $QWEN | head -20; ls $PHO | head -20
# bộ chấm + test đơn vị (cần data đã gắn)
python -X utf8 scripts/verify_metric.py 2>&1 | tail -5
python -m pytest tests -q 2>&1 | tail -3

In [ ]:
%%bash
# dry run: in lệnh sẽ chạy, không chạy gì
source /kaggle/working/lifemate/kaggle_env.sh
DRYRUN=1 bash scripts/kaggle_pipeline.sh asr_base | head -6
DRYRUN=1 bash scripts/kaggle_pipeline.sh train

In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh
SMOKE=1 bash scripts/kaggle_pipeline.sh asr_base          # ASR 4 GPU song song, 16 wav mỗi shard

In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh
# 30 bước SFT trên 3 GPU (+ fine-tune ASR 30 bước trên GPU cuối); nhìn dòng giây/bước để tính thời gian thật
SMOKE=1 SMOKE_STEPS=30 bash scripts/kaggle_pipeline.sh train 2>&1 | grep -E "step examples|trainable|it/s|s/it|saved|Error|Traceback|train_runtime"

In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh
SMOKE=1 bash scripts/kaggle_pipeline.sh asr_ft            # ASR đã fine-tune (smoke), 4 GPU
SMOKE=1 bash scripts/kaggle_pipeline.sh eval_dev          # suy luận 8 mẫu dev với adapter smoke (điểm vô nghĩa, chỉ kiểm tra ống dẫn)

In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh || { echo "ERROR: chưa có kaggle_env.sh, chạy cell setup trước"; exit 1; }
echo "số GPU: $(python -c 'import torch; print(torch.cuda.device_count())')"
if bash scripts/infer_multi_gpu.sh train outputs/dev_q4b_clean.jsonl --ids data/processed/dev_ids.txt --backend hf --model $QWEN --text clean; then
  python -X utf8 scripts/eval.py --sub outputs/dev_q4b_clean.jsonl --ids data/processed/dev_ids.txt
  tail -n 1 outputs/dev_q4b_clean.part*.log        # mean / p95 / max giây mỗi mẫu
else
  echo "=== INFER LỖI. Phần cuối log từng shard: ==="
  ls outputs/ | head -20
  for f in outputs/dev_q4b_clean.part*.log; do echo "--- $f"; tail -n 25 "$f"; done
  exit 1
fi

A + B tốt

In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh
bash scripts/kaggle_pipeline.sh asr_base          # ASR train + public_test + WER (kỳ vọng ~26,2%)
bash scripts/kaggle_pipeline.sh sft_asr_data      # tạo sft_train_asr.jsonl

In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh
# SFT (3 GPU) + fine-tune ASR (GPU cuối). EPOCHS_SFT và MAX_HOURS chỉnh theo tốc độ đo ở phần B.
EPOCHS_SFT=3 MAX_HOURS=10.5 bash scripts/kaggle_pipeline.sh train

In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh
bash scripts/kaggle_pipeline.sh eval_dev          # điểm dev với adapter, text = ASR gốc

In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh
bash scripts/kaggle_pipeline.sh asr_ft            # chỉ dùng bản fine-tune nếu WER dev giảm rõ so với ~25,4%
bash scripts/kaggle_pipeline.sh sft_ft_data

In [ ]:
%%bash
# Lưu trạng thái cho phiên sau (xóa bản làm việc lifemate/), rồi bấm Save Version.
bash /kaggle/working/lifemate/scripts/kaggle_save_state.sh